# 1. Setup and Data Loading

In [1]:
# Standard libraries
import json
from pathlib import Path
from collections import Counter

# Data handling
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt

# Notebook settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

DATA_PATH = Path("../data/raw")

In [2]:
# Load schema

with open(DATA_PATH / "candidate_schema.json", "r", encoding="utf-8") as f:
    schema = json.load(f)

print(type(schema))
print(schema.keys())

<class 'dict'>
dict_keys(['$schema', 'title', 'description', 'type', 'required', 'properties'])


In [3]:
print("Schema Title:")
print(schema["title"])

print("\nDescription:")
print(schema["description"])

print("\nTop Level Type:")
print(schema["type"])

print("\nRequired Fields:")
print(schema["required"])

Schema Title:
Redrob Candidate Profile Schema

Description:
Schema for a single candidate profile in the Intelligent Candidate Discovery & Ranking Challenge dataset.

Top Level Type:
object

Required Fields:
['candidate_id', 'profile', 'career_history', 'education', 'skills', 'redrob_signals']


In [4]:
properties = schema["properties"]

print("Number of top-level fields:", len(properties))
print("\nTop-level fields:\n")

for field in properties.keys():
    print("-", field)

Number of top-level fields: 8

Top-level fields:

- candidate_id
- profile
- career_history
- education
- skills
- certifications
- languages
- redrob_signals


In [5]:
for field, details in properties.items():
    print("\n" + "="*70)
    print("FIELD:", field)

    if isinstance(details, dict):
        print("Type:", details.get("type"))

        if "description" in details:
            print("Description:", details["description"])

        if "properties" in details:
            print("Nested fields:")
            print(list(details["properties"].keys()))

        if "items" in details:
            print("Array structure detected")


FIELD: candidate_id
Type: string
Description: Unique identifier for the candidate. Format: CAND_XXXXXXX (7 digits).

FIELD: profile
Type: object
Nested fields:
['anonymized_name', 'headline', 'summary', 'location', 'country', 'years_of_experience', 'current_title', 'current_company', 'current_company_size', 'current_industry']

FIELD: career_history
Type: array
Array structure detected

FIELD: education
Type: array
Array structure detected

FIELD: skills
Type: array
Array structure detected

FIELD: certifications
Type: array
Array structure detected

FIELD: languages
Type: array
Array structure detected

FIELD: redrob_signals
Type: object
Description: Simulated platform activity and engagement signals from the Redrob ecosystem.
Nested fields:
['profile_completeness_score', 'signup_date', 'last_active_date', 'open_to_work_flag', 'profile_views_received_30d', 'applications_submitted_30d', 'recruiter_response_rate', 'avg_response_time_hours', 'skill_assessment_scores', 'connection_count'

In [6]:
# Load first candidate from candidates.jsonl

with open(DATA_PATH / "candidates.jsonl", "r", encoding="utf-8") as f:
    first_candidate = json.loads(next(f))

print(type(first_candidate))
print(first_candidate.keys())

<class 'dict'>
dict_keys(['candidate_id', 'profile', 'career_history', 'education', 'skills', 'certifications', 'languages', 'redrob_signals'])


In [7]:
for key, value in first_candidate.items():
    print("\n" + "="*80)
    print(f"FIELD: {key}")
    print(f"DATA TYPE: {type(value)}")

    if isinstance(value, dict):
        print("NESTED KEYS:")
        print(list(value.keys()))

    elif isinstance(value, list):
        print(f"NUMBER OF ITEMS: {len(value)}")

        if len(value) > 0:
            print("\nFIRST ITEM:")
            print(value[0])

    else:
        print("VALUE:")
        print(value)


FIELD: candidate_id
DATA TYPE: <class 'str'>
VALUE:
CAND_0000001

FIELD: profile
DATA TYPE: <class 'dict'>
NESTED KEYS:
['anonymized_name', 'headline', 'summary', 'location', 'country', 'years_of_experience', 'current_title', 'current_company', 'current_company_size', 'current_industry']

FIELD: career_history
DATA TYPE: <class 'list'>
NUMBER OF ITEMS: 2

FIRST ITEM:
{'company': 'Mindtree', 'title': 'Backend Engineer', 'start_date': '2024-03-08', 'end_date': None, 'duration_months': 27, 'is_current': True, 'industry': 'IT Services', 'company_size': '10001+', 'description': 'Implemented streaming data pipelines on Kafka and Spark Streaming for a real-time user-activity processing platform. Designed the schema-registry integration, the watermark/state management approach, and the deduplication logic for late-arriving events. Worked closely with the data science team to make sure feature pipelines aligned with what their models needed. Most of my career has been data engineering, with so

In [8]:
# Count total candidates

candidate_count = 0

with open(DATA_PATH / "candidates.jsonl", "r", encoding="utf-8") as f:
    for _ in f:
        candidate_count += 1

print("Total Candidates:", candidate_count)

Total Candidates: 100000


In [9]:
# Load first 1000 candidates for exploration

sample_candidates = []

with open(DATA_PATH / "candidates.jsonl", "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 1000:
            break

        sample_candidates.append(json.loads(line))

print("Loaded:", len(sample_candidates))

Loaded: 1000


In [10]:
# Inspect random candidate titles

import random

for _ in range(10):
    c = random.choice(sample_candidates)

    print(
        c["candidate_id"],
        "|",
        c["profile"]["current_title"]
    )

CAND_0000592 | .NET Developer
CAND_0000308 | Mechanical Engineer
CAND_0000168 | Sales Executive
CAND_0000512 | Project Manager
CAND_0000326 | Full Stack Developer
CAND_0000164 | Content Writer
CAND_0000820 | Backend Engineer
CAND_0000948 | .NET Developer
CAND_0000412 | Marketing Manager
CAND_0000301 | Operations Manager


In [11]:
# Extract titles from first 5000 candidates

titles = []

with open(DATA_PATH / "candidates.jsonl", "r", encoding="utf-8") as f:

    for i, line in enumerate(f):

        if i >= 5000:
            break

        candidate = json.loads(line)

        title = candidate["profile"].get("current_title")

        if title:
            titles.append(title)

title_counts = Counter(titles)

print("Unique Titles:", len(title_counts))

print("\nTop 30 Titles:\n")

for title, count in title_counts.most_common(30):
    print(f"{title}: {count}")

Unique Titles: 37

Top 30 Titles:

HR Manager: 304
Sales Executive: 303
Mechanical Engineer: 301
Business Analyst: 299
Accountant: 295
Civil Engineer: 289
Project Manager: 287
Graphic Designer: 282
Marketing Manager: 276
Operations Manager: 275
Content Writer: 272
Customer Support: 266
Mobile Developer: 171
Software Engineer: 159
QA Engineer: 150
Frontend Engineer: 146
DevOps Engineer: 140
Full Stack Developer: 140
Java Developer: 133
Cloud Engineer: 117
.NET Developer: 115
Analytics Engineer: 42
Data Analyst: 39
Senior Data Engineer: 39
Data Engineer: 35
Senior Software Engineer: 34
Backend Engineer: 33
ML Engineer: 12
Senior Software Engineer (ML): 10
Data Scientist: 8


In [12]:
# Check technical vs non-technical titles

tech_keywords = [
    "engineer", "developer", "scientist",
    "analyst", "architect", "devops",
    "ml", "data", "qa", "backend",
    "frontend", "full stack", "cloud"
]

tech_count = 0
non_tech_count = 0

for title in titles:
    title_lower = title.lower()

    if any(k in title_lower for k in tech_keywords):
        tech_count += 1
    else:
        non_tech_count += 1

print("Technical Titles:", tech_count)
print("Non-Technical Titles:", non_tech_count)
print("Technical %:", round(100 * tech_count / len(titles), 2))

Technical Titles: 2433
Non-Technical Titles: 2567
Technical %: 48.66


In [13]:
skill_counter = Counter()

with open(DATA_PATH / "candidates.jsonl", "r", encoding="utf-8") as f:

    for i, line in enumerate(f):

        if i >= 10000:   # first 10k candidates
            break

        candidate = json.loads(line)

        for skill in candidate.get("skills", []):

            name = skill.get("name")

            if name:
                skill_counter[name.strip().lower()] += 1

print("Unique Skills:", len(skill_counter))

print("\nTop 50 Skills:\n")

for skill, count in skill_counter.most_common(50):
    print(f"{skill}: {count}")

Unique Skills: 129

Top 50 Skills:

html: 1263
content writing: 1262
react: 1259
databricks: 1257
redux: 1253
css: 1250
kafka: 1243
spring boot: 1240
seo: 1238
javascript: 1232
grpc: 1231
redis: 1229
aws: 1227
scrum: 1224
spark: 1221
data pipelines: 1221
marketing: 1217
azure: 1217
apache flink: 1215
django: 1214
vue.js: 1214
agile: 1214
accounting: 1212
salesforce crm: 1210
excel: 1207
illustrator: 1206
rest apis: 1206
java: 1205
typescript: 1204
sales: 1203
dbt: 1203
bigquery: 1202
docker: 1202
gcp: 1199
angular: 1198
tally: 1198
photoshop: 1195
ci/cd: 1195
go: 1194
project management: 1191
rust: 1190
airflow: 1189
flask: 1188
webpack: 1188
postgresql: 1188
apache beam: 1183
sap: 1183
mongodb: 1183
fastapi: 1183
kubernetes: 1180


In [15]:
import sys
from pathlib import Path

# Adds the parent directory to your path dynamically
sys.path.append(str(Path.cwd().parent))

from collections import Counter
import json
from pathlib import Path

from src.preprocessing.role_classifier import classify_role

DATA_PATH = Path("../data/raw")   # adjust if needed

counts = Counter()

with open(DATA_PATH / "candidates.jsonl", "r", encoding="utf-8") as f:

    for i, line in enumerate(f):

        if i >= 1000:
            break

        candidate = json.loads(line)

        title = (
            candidate.get("profile", {})
                     .get("current_title", "")
        )

        role_type = classify_role(title)

        counts[role_type] += 1

print(counts)

Counter({'non_technical': 444, 'technical': 304, 'unknown': 252})
